<a href="https://colab.research.google.com/github/alokshah04/pgss-L4R-RL-public/blob/main/student_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforcement Learning: Teaching a Robot by Trial and Error

In this lab you will train a **HalfCheetah** robot to run using **Proximal Policy Optimization (PPO)** — a popular RL algorithm. Unlike Behavior Cloning, we provide **no expert demonstrations**: the agent learns entirely from rewards it earns by interacting with the environment.

The pipeline has four repeating steps:

```
1. COLLECT  — run the current policy in the environment; store (obs, action, reward, done)
2. ESTIMATE — compute advantages using Generalized Advantage Estimation (GAE)
3. UPDATE   — improve the policy with multiple epochs of clipped gradient steps
4. REPEAT
```

---

## ⚙️ Setup

### 💻 Running locally
```bash
conda activate il-lab   # same environment as the IL lab
```
Then launch Jupyter from the repo folder. **Skip Step 0.**

### ☁️ Running on Google Colab
**1.** Switch to a T4 GPU: Runtime → Change runtime type → T4 GPU → Save  
**2.** Run Step 0 first and wait for `✅ Setup complete`.

> ⚠️ Video cells (Steps 1a, 1b, 9) do not work on Colab — skip those.

---

## 📋 How to use this notebook

Sections marked `# TODO` are yours to implement. You are encouraged to use AI tools, but **every line of AI-generated code must have a comment written by you** explaining what it does and why.

**Expected training time:** Colab T4 GPU ~15 min · Laptop CPU ~30–40 min  
**Goal:** eval return ≥ 1000 at 2M environment steps.

---
## Step 0 — Colab Setup ☁️

> **Local users: skip this cell.**

In [ ]:
# ════════════════════════════════════════════════
# COLAB ONLY — local users skip this cell
# ════════════════════════════════════════════════
import subprocess, sys, os

REPO = 'pgss-L4R-RL'
if not os.path.isdir(REPO):
    r = subprocess.run(['git', 'clone', 'https://github.com/alokshah04/pgss-L4R-RL-public.git'],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print('Repo already cloned.')
if not os.getcwd().endswith(REPO):
    os.chdir(REPO)
print('Working directory:', os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('\n✅ Setup complete. Continue to Step 1.')

---
## Step 1 — Imports 💻☁️

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import Video, display

ENV_NAME = 'HalfCheetah-v5'

env = gym.make(ENV_NAME)
OBS_DIM = env.observation_space.shape[0]   # 17
ACT_DIM = env.action_space.shape[0]        # 6
ACT_HIGH = float(env.action_space.high[0]) # 1.0
env.close()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'obs_dim = {OBS_DIM},  act_dim = {ACT_DIM},  act_bound = ±{ACT_HIGH}')
print(f'Device: {DEVICE}')

### Step 1a — Video recorder 💻 (local only)

> **Colab users: skip Steps 1a and 1b.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
import imageio

def record_policy(step_fn, env_name, path, max_steps=500, seed=1, fps=30):
    """Run step_fn(obs) -> action and save an MP4."""
    env = gym.make(env_name, render_mode='rgb_array')
    obs, _ = env.reset(seed=seed)
    frames, total_reward = [], 0.0
    for _ in range(max_steps):
        action = step_fn(obs)
        obs, reward, done, truncated, _ = env.step(action)
        total_reward += reward
        frames.append(env.render())
        if done or truncated:
            break
    env.close()
    imageio.mimsave(path, frames, fps=fps)
    print(f'Saved {path}  |  reward: {total_reward:.1f}  |  frames: {len(frames)}')
    return total_reward

print('record_policy defined.')

### Step 1b — Watch an untrained policy 💻 (local only)

> **Colab users: skip this cell.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
def random_step(obs):
    return np.random.uniform(-1, 1, size=(ACT_DIM,)).astype(np.float32)

record_policy(random_step, ENV_NAME, 'random_policy.mp4')
display(Video('random_policy.mp4', embed=True, width=600))

---
## Step 2 — Hyperparameters 💻☁️

These control how training runs. We will refer back to them throughout.

**Your turn:** read each comment and think about what would happen if you doubled or halved each value.

In [ ]:
# ── Rollout collection ───────────────────────────────────────────────────────
NUM_ENVS      = 4       # TODO: how many parallel environments should we run?
ROLLOUT_STEPS = 2048    # steps collected per env per rollout
TOTAL_STEPS   = 2_000_000

# ── PPO update ───────────────────────────────────────────────────────────────
EPOCHS_PER_ROLLOUT = 10     # how many passes over each rollout's data
MINIBATCH_SIZE     = 256
CLIP_EPS           = 0.2    # TODO: what does this control in PPO?
VALUE_LOSS_COEF    = 0.5
ENTROPY_COEF       = 0.0
MAX_GRAD_NORM      = 0.5

# ── Return estimation ────────────────────────────────────────────────────────
GAMMA      = 0.99   # discount factor
GAE_LAMBDA = 0.95   # TODO: what does GAE_LAMBDA control?

# ── Optimizer ────────────────────────────────────────────────────────────────
LR        = 3e-4
ANNEAL_LR = True    # linearly decay learning rate to 0

# ── Network ──────────────────────────────────────────────────────────────────
HIDDEN = 256

# ── Logging ──────────────────────────────────────────────────────────────────
LOG_INTERVAL  = 5
EVAL_INTERVAL = 20
EVAL_EPISODES = 5

total_rollouts = TOTAL_STEPS // (ROLLOUT_STEPS * NUM_ENVS)
print(f'Total rollouts: {total_rollouts}  ({TOTAL_STEPS/1e6:.1f}M env steps)')
print(f'Data per rollout: {ROLLOUT_STEPS * NUM_ENVS:,} steps')

---
## Step 3 — The Actor-Critic Network 💻☁️

PPO maintains **two outputs** from a shared neural network trunk:

- **Actor (policy):** outputs the *mean* of a Gaussian distribution over actions. We sample from this during training (exploration) and use the mean during evaluation (exploitation).
- **Critic (value function):** outputs a scalar $V(s)$ — the expected discounted return from state $s$.

```
obs (17) → [Linear → LayerNorm → Tanh] × 3
                          ↓
             actor_mean (6)    critic (1)
```

The **log standard deviation** is a learned free parameter (not observation-dependent) initialized to 0, so $\sigma = e^0 = 1$ — wide exploration at the start.

**Why Tanh?** It squashes activations to $(-1, 1)$, which helps stabilize the critic when value targets grow large.

### 📄 Citation Task

Find **one paper** that justifies using orthogonal initialization in deep RL actor-critic networks. Include the BibTeX below.

### BibTeX Citation

**Orthogonal initialization** — paste your citation and a one-sentence justification here:

```bibtex
% TODO: paste BibTeX here
```

*Justification (your own words):*

In [ ]:
class ActorCritic(nn.Module):
    """
    Shared-trunk actor-critic network for PPO.
    Actor outputs a Gaussian mean; critic outputs a scalar value estimate.
    """
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        # TODO: build self.trunk as an nn.Sequential with 3 hidden layers,
        # each consisting of: Linear -> LayerNorm -> Tanh
        self.trunk = None  # replace with nn.Sequential(...)

        # TODO: define self.actor_mean as a Linear layer (hidden -> act_dim)
        self.actor_mean = None

        # TODO: define self.critic as a Linear layer (hidden -> 1)
        self.critic = None

        # log_std is a free parameter (not obs-dependent).
        # Initializing to 0 means std=1 at the start — wide exploration.
        self.log_std = nn.Parameter(torch.zeros(act_dim))

        # TODO: apply orthogonal initialization to all Linear layers.
        # Use gain=sqrt(2) for trunk layers, gain=0.01 for actor_mean,
        # gain=1.0 for critic, and zero-initialize all biases.
        # (Hint: iterate over self.modules() and check isinstance(m, nn.Linear))
        pass

    def forward(self, obs):
        """
        TODO: pass obs through self.trunk, then return:
          - actor_mean(features)
          - critic(features).squeeze(-1)   ← squeeze to remove the trailing 1
        """
        pass

    def get_distribution(self, obs):
        """
        TODO: call self.forward(obs) to get the mean, then construct a
        Normal distribution with std = exp(self.log_std).
        Return the distribution object.

        Hint: use self.log_std.exp().expand_as(mean) to match the mean's shape.
        """
        pass

    def act(self, obs):
        """
        TODO: Sample an action from the distribution and return
        (action, log_prob, value).

        - action   = dist.sample()
        - log_prob = dist.log_prob(action).sum(-1)  ← sum over action dims
        - value    = from self.forward(obs)
        """
        pass

    def evaluate(self, obs, action):
        """
        Evaluate stored actions under the CURRENT policy. Used during PPO update.

        TODO: return (log_prob, entropy, value) where:
        - log_prob = dist.log_prob(action).sum(-1)
        - entropy  = dist.entropy().sum(-1)   ← measures exploration
        - value    = from self.forward(obs)
        """
        pass


# Quick sanity check
test_model = ActorCritic(OBS_DIM, ACT_DIM, HIDDEN).to(DEVICE)
test_obs   = torch.zeros(1, OBS_DIM, device=DEVICE)
mean, val  = test_model(test_obs)
print(f'actor_mean shape: {mean.shape}  (should be [1, {ACT_DIM}])')
print(f'value shape:      {val.shape}   (should be [1])')
print(f'Total parameters: {sum(p.numel() for p in test_model.parameters()):,}')
del test_model

---
## Step 4 — Rollout Buffer 💻☁️

A **rollout buffer** stores everything the agent experienced during one collection phase:

| Field | Shape | What it stores |
|-------|-------|----------------|
| `obs` | (T, N, obs_dim) | observations |
| `actions` | (T, N, act_dim) | sampled actions |
| `rewards` | (T, N) | rewards received |
| `dones` | (T, N) | episode-end flags |
| `log_probs` | (T, N) | log π(a|s) at collection time |
| `values` | (T, N) | critic estimates V(s) |

T = timesteps per rollout, N = number of parallel environments.

### Generalized Advantage Estimation (GAE)

The **advantage** $A_t$ measures how much better action $a_t$ was than average in state $s_t$.
GAE computes it as an exponentially-weighted sum of **TD errors**:

$$\delta_t = r_t + \gamma V(s_{t+1})(1 - d_t) - V(s_t)$$

$$A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

In practice we compute this **backwards** in a single pass:

$$A_t = \delta_t + \gamma \lambda (1 - d_t) A_{t+1}$$

The **return** (target for the critic) is:
$$R_t = A_t + V(s_t)$$

In [ ]:
class RolloutBuffer:
    def __init__(self, steps, num_envs, obs_dim, act_dim, device):
        self.T, self.N = steps, num_envs
        self.device = device
        # TODO: pre-allocate tensors for obs, actions, rewards, dones,
        # log_probs, and values. Each should have the shape shown in the table above.
        # Use torch.zeros(..., device=device) for each.
        self.obs       = None  # shape (steps, num_envs, obs_dim)
        self.actions   = None  # shape (steps, num_envs, act_dim)
        self.rewards   = None  # shape (steps, num_envs)
        self.dones     = None  # shape (steps, num_envs)
        self.log_probs = None  # shape (steps, num_envs)
        self.values    = None  # shape (steps, num_envs)
        self.ptr = 0

    def store(self, obs, action, reward, done, log_prob, value):
        # TODO: store each argument at position self.ptr, then increment self.ptr
        pass

    def compute_gae(self, next_value, next_done):
        """
        Compute GAE advantages and returns, iterating BACKWARDS through time.

        TODO: implement the backwards recursion:
          advantages = zeros_like(self.rewards)
          last_adv = 0.0
          for t in reversed(range(self.T)):
              if t == self.T - 1:
                  nv = next_value   (bootstrapped from the value after the rollout)
                  nd = next_done
              else:
                  nv = self.values[t + 1]
                  nd = self.dones[t + 1]
              delta    = self.rewards[t] + GAMMA * nv * (1 - nd) - self.values[t]
              last_adv = delta + GAMMA * GAE_LAMBDA * (1 - nd) * last_adv
              advantages[t] = last_adv
          returns = advantages + self.values
          return advantages, returns
        """
        pass

    def get_minibatches(self, advantages, returns):
        """
        Flatten (T, N, ...) to (T*N, ...) and yield shuffled minibatches.

        TODO:
        1. Flatten obs, actions, log_probs, values, advantages, returns
           (obs: .view(-1, obs_dim), actions: .view(-1, act_dim), others: .view(-1))
        2. Normalize advantages: adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        3. Shuffle with torch.randperm, then yield MINIBATCH_SIZE slices of
           (obs[idx], actions[idx], log_probs[idx], values[idx], adv[idx], ret[idx])
        """
        pass

    def reset(self):
        self.ptr = 0


# Sanity check
test_buf = RolloutBuffer(ROLLOUT_STEPS, NUM_ENVS, OBS_DIM, ACT_DIM, DEVICE)
print(f'obs buffer shape:    {test_buf.obs.shape}     (should be [{ROLLOUT_STEPS}, {NUM_ENVS}, {OBS_DIM}])')
print(f'actions buffer shape: {test_buf.actions.shape}  (should be [{ROLLOUT_STEPS}, {NUM_ENVS}, {ACT_DIM}])')
print(f'rewards buffer shape: {test_buf.rewards.shape}  (should be [{ROLLOUT_STEPS}, {NUM_ENVS}])')
del test_buf

---
## Step 5 — Vectorized Environments 💻☁️

Running `NUM_ENVS` environments in parallel multiplies data throughput without extra wall time. After each step, if an environment is done, we **auto-reset** it and continue collecting.

In [ ]:
class VecEnv:
    def __init__(self, env_name, num_envs, seed=0):
        self.envs = [gym.make(env_name) for _ in range(num_envs)]
        self.num_envs = num_envs
        for i, e in enumerate(self.envs):
            e.reset(seed=seed + i)

    def reset(self):
        # TODO: reset all envs and return a (num_envs, obs_dim) float32 numpy array
        pass

    def step(self, actions):
        """
        Step all envs simultaneously.

        TODO: for each (env, action) pair:
        - step the env
        - if done (terminated or truncated), auto-reset the env and use the reset obs
        - collect obs, reward, done into lists
        Return (obs_array, reward_array, done_array) as float32 numpy arrays.
        """
        pass

    def close(self):
        for e in self.envs: e.close()

print('VecEnv defined.')

---
## Step 6 — Evaluation Helper 💻☁️

During evaluation we use the **deterministic** policy (mean action, no sampling) so results are not confounded by exploration noise.

In [ ]:
def evaluate(policy, env_name, num_episodes=5, max_steps=1000, seed=999):
    """Run the deterministic policy and return (mean_return, std_return)."""
    policy.eval()
    returns = []
    for ep in range(num_episodes):
        env = gym.make(env_name)
        obs, _ = env.reset(seed=seed + ep)
        ep_ret = 0.0
        with torch.no_grad():
            for _ in range(max_steps):
                # TODO: convert obs to a tensor, get the mean action from the policy
                # (use policy(obs_t) and take the first return — the mean),
                # convert to numpy, step the env, accumulate ep_ret, break if done
                pass
        env.close()
        returns.append(ep_ret)
    policy.train()
    return np.mean(returns), np.std(returns)

print('evaluate() defined.')

---
## Step 7 — The PPO Loss 💻☁️

PPO improves the policy using two losses:

### Policy loss (clipped surrogate)

Define the **probability ratio** between new and old policy:
$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)} = \exp(\log\pi_\theta(a_t|s_t) - \log\pi_{\theta_{\text{old}}}(a_t|s_t))$$

PPO clips this ratio so updates cannot change the policy too aggressively:
$$\mathcal{L}^{\text{CLIP}} = -\mathbb{E}_t\!\left[\min\!\left(r_t A_t,\ \text{clip}(r_t, 1{-}\epsilon, 1{+}\epsilon)\, A_t\right)\right]$$

### Value loss

$$\mathcal{L}^{\text{VF}} = \frac{1}{2}\, \mathbb{E}_t\!\left[\left(V_\theta(s_t) - R_t\right)^2\right]$$

### Total loss
$$\mathcal{L} = \mathcal{L}^{\text{CLIP}} + c_v \mathcal{L}^{\text{VF}}$$

where $c_v = $ `VALUE_LOSS_COEF`.

In [ ]:
def ppo_update(policy, optimizer, buffer, advantages, returns):
    """Run EPOCHS_PER_ROLLOUT passes of PPO updates over the current rollout."""
    pl_log, vl_log, ent_log = [], [], []

    for _ in range(EPOCHS_PER_ROLLOUT):
        for obs_b, act_b, lp_old, val_old, adv_b, ret_b in \
                buffer.get_minibatches(advantages, returns):

            # Evaluate stored actions under the CURRENT policy
            lp_new, ent, val_new = policy.evaluate(obs_b, act_b)

            # ── Policy loss ──────────────────────────────────────────────────
            # TODO: compute ratio = exp(lp_new - lp_old)
            # TODO: compute pg_loss1 = -adv_b * ratio  (unclipped)
            # TODO: compute pg_loss2 = -adv_b * ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS)  (clipped)
            # TODO: policy_loss = torch.max(pg_loss1, pg_loss2).mean()
            #       (we take the MAX because both terms are negated for gradient ascent)
            policy_loss = None  # replace

            # ── Value loss ───────────────────────────────────────────────────
            # TODO: compute the MSE between val_new and ret_b
            # Optionally: also clip the value update (same idea as policy clipping):
            #   val_clipped = val_old + (val_new - val_old).clamp(-CLIP_EPS, CLIP_EPS)
            #   value_loss  = 0.5 * torch.max((val_new - ret_b)**2,
            #                                  (val_clipped - ret_b)**2).mean()
            value_loss = None  # replace

            # ── Combined loss ────────────────────────────────────────────────
            # TODO: loss = policy_loss + VALUE_LOSS_COEF * value_loss
            # TODO: zero gradients, backprop, clip gradients (MAX_GRAD_NORM), optimizer step
            pass

            pl_log.append(policy_loss.item())
            vl_log.append(value_loss.item())
            ent_log.append(ent.mean().item())

    return np.mean(pl_log), np.mean(vl_log), np.mean(ent_log)

print('ppo_update() defined.')

---
## Step 8 — Train! 💻☁️

The outer loop repeats collect → GAE → update until we reach `TOTAL_STEPS`.

**Expected wall time:** ~15 min (Colab T4) / ~30 min (CPU)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

policy    = ActorCritic(OBS_DIM, ACT_DIM, HIDDEN).to(DEVICE)
optimizer = optim.Adam(policy.parameters(), lr=LR, eps=1e-5)
buffer    = RolloutBuffer(ROLLOUT_STEPS, NUM_ENVS, OBS_DIM, ACT_DIM, DEVICE)
vec_env   = VecEnv(ENV_NAME, NUM_ENVS, seed=42)

obs  = torch.tensor(vec_env.reset(), dtype=torch.float32, device=DEVICE)
done = torch.zeros(NUM_ENVS, device=DEVICE)

episode_returns = []
eval_steps      = []
eval_means      = []
eval_stds       = []
policy_losses   = []
value_losses    = []
steps_done      = 0
current_ep_ret  = torch.zeros(NUM_ENVS, device=DEVICE)

pbar = tqdm(range(total_rollouts), desc='PPO')
for rollout_idx in pbar:

    # ── LR annealing ──────────────────────────────────────────────────────
    if ANNEAL_LR:
        frac = 1.0 - rollout_idx / total_rollouts
        for g in optimizer.param_groups:
            g['lr'] = LR * frac

    # ── Collect rollout ────────────────────────────────────────────────────
    buffer.reset()
    policy.eval()
    rollout_ep_returns = []

    with torch.no_grad():
        for step in range(ROLLOUT_STEPS):
            # TODO: call policy.act(obs) to get (action, log_prob, value)
            # TODO: convert action to numpy and step the vec_env
            # TODO: convert reward_np and next_obs_np to tensors on DEVICE
            # TODO: call buffer.store(obs, action, reward, done, log_prob, value)
            # TODO: track episode returns: add reward to current_ep_ret;
            #       when done_np[i] is True, append current_ep_ret[i] to
            #       rollout_ep_returns and reset current_ep_ret[i] to 0
            # TODO: update obs and done for the next step
            pass

        # Bootstrap value for the last obs (used in GAE)
        # TODO: get next_value from policy.act(obs)[2]
        next_value = None

    # TODO: call buffer.compute_gae(next_value, done) to get advantages, returns
    advantages, returns = None, None
    steps_done += ROLLOUT_STEPS * NUM_ENVS

    # ── PPO update ─────────────────────────────────────────────────────────
    policy.train()
    # TODO: call ppo_update(policy, optimizer, buffer, advantages, returns)
    # and unpack the returned (pl, vl, ent)
    pl, vl, ent = None, None, None
    policy_losses.append(pl)
    value_losses.append(vl)

    if rollout_ep_returns:
        episode_returns.append(np.mean(rollout_ep_returns))

    # ── Logging ────────────────────────────────────────────────────────────
    if (rollout_idx + 1) % LOG_INTERVAL == 0:
        avg = np.mean(episode_returns[-20:]) if episode_returns else float('nan')
        pbar.set_postfix({'steps': f'{steps_done/1e6:.2f}M', 'ret': f'{avg:.0f}',
                          'pl': f'{pl:.4f}', 'vl': f'{vl:.2f}'})

    if (rollout_idx + 1) % EVAL_INTERVAL == 0:
        m, s = evaluate(policy, ENV_NAME, num_episodes=EVAL_EPISODES)
        eval_means.append(m); eval_stds.append(s); eval_steps.append(steps_done)
        print(f'\n[{steps_done/1e6:.2f}M steps] eval: {m:.1f} ± {s:.1f}')

vec_env.close()
print('\nTraining complete!')
torch.save(policy.state_dict(), 'ppo_policy.pt')
print('Saved ppo_policy.pt')

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(episode_returns, alpha=0.3, label='episode return')
W = 20
if len(episode_returns) >= W:
    smooth = np.convolve(episode_returns, np.ones(W)/W, mode='valid')
    axes[0].plot(range(W-1, len(episode_returns)), smooth, label=f'{W}-ep avg')
axes[0].set_xlabel('Completed episodes'); axes[0].set_ylabel('Return')
axes[0].set_title('Training Return (stochastic)'); axes[0].legend()

axes[1].errorbar(eval_steps, eval_means, yerr=eval_stds, fmt='o-', capsize=4)
axes[1].set_xlabel('Environment steps'); axes[1].set_ylabel('Eval return')
axes[1].set_title('Evaluation Return (deterministic)')

axes[2].plot(policy_losses, label='policy loss')
axes[2].plot(value_losses,  label='value loss')
axes[2].set_xlabel('Rollout'); axes[2].set_ylabel('Loss')
axes[2].set_title('PPO Losses'); axes[2].legend()

plt.tight_layout()
plt.savefig('ppo_training.png', dpi=120)
plt.show()

---
## Step 9 — Watch Your Robot Run! 💻 (local only)

> **Colab users: skip this cell.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
policy.eval()

def ppo_step(obs):
    with torch.no_grad():
        obs_t   = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        mean, _ = policy(obs_t)
        return mean.squeeze(0).cpu().numpy()

print('--- Untrained (random) ---')
record_policy(random_step, ENV_NAME, 'random_policy.mp4')
display(Video('random_policy.mp4', embed=True, width=600))

print('\n--- PPO-trained policy ---')
record_policy(ppo_step, ENV_NAME, 'ppo_policy.mp4')
display(Video('ppo_policy.mp4', embed=True, width=600))

---
## Step 10 — Wait... It's Not Running 🤔

If you watched the video above, you may have noticed something odd: **the cheetah isn't actually running**. It moves forward, but it does so by flopping on its back and wiggling its legs in the air.

The reward is still positive — so what's going on?

### The reward function

Look up the HalfCheetah-v5 reward specification in the Gymnasium docs:  
**https://gymnasium.farama.org/environments/mujoco/half_cheetah/**

The reward is defined as:

$$r_t = v_x - c \cdot \|a_t\|^2$$

where $v_x$ is the **forward velocity of the torso** and $c$ is a small control cost penalty.

There is **no term that rewards staying upright**, no penalty for being on its back, and no requirement that the legs touch the ground in any particular way.

### What happened

The agent discovered that flopping onto its back and thrusting its legs forward is a perfectly valid strategy under this reward. It earns positive $v_x$ without ever learning to run. The reward went up — the agent did its job — but the behavior is not what we intended.

This phenomenon has a name: **reward hacking** (also called specification gaming). The agent finds an unintended solution that maximizes the reward function as written, rather than the behavior the designer had in mind.

> *"You get what you measure."* — Goodhart's Law

### Your turn

Write answers to these questions in the cell below before moving on.

1. **Why did this happen?** In one or two sentences, explain what reward hacking is and why the cheetah's backflop strategy is a valid solution under this reward.

2. **How would you fix it?** Propose at least **two** changes to the reward function that would discourage this behavior. For each one, explain what new incentive it introduces and whether it could create any new unintended behaviors.

   Some things to consider:
   - What observations does the cheetah have access to? (Check the obs table in the Gymnasium docs — torso height and angle are in there.)
   - Could you penalize the torso being below a certain height?
   - Could you reward foot contact with the ground?
   - What might go wrong with each of your proposals?

3. **Is this a PPO problem?** Would this issue disappear if you used a different RL algorithm (e.g. SAC, TD3)? Why or why not?

### Your answers

**(1) Why did this happen?**

*TODO: your answer here*

---

**(2) How would you fix it?**

| Proposal | What incentive it adds | Potential new problems |
|----------|------------------------|------------------------|
| | | |
| | | |

---

**(3) Is this a PPO problem?**

*TODO: your answer here*

---
## Discussion Questions

Answer these in writing before submitting.

1. **RL vs IL.** In the IL lab you trained from expert demonstrations; here you use only rewards. What is the fundamental tradeoff between these two approaches?

2. **The learning curve.** Why does return start negative (or near zero)? Why does it rise quickly at first and then slow down?

3. **Clipping.** Why does PPO clip the probability ratio to $[1-\epsilon, 1+\epsilon]$? What would happen if you set $\epsilon = \infty$ (no clip)?

4. **GAE.** `GAE_LAMBDA` blends between pure TD errors ($\lambda=0$) and Monte Carlo returns ($\lambda=1$). What is the bias-variance tradeoff between these extremes?

5. **Value function role.** The critic is never deployed — only the actor runs at test time. Why do we train a critic at all?

6. **Exploration.** How does the policy's standard deviation evolve during training, and why?

7. **AI use reflection.** Which parts did you use AI for? For one specific piece of AI-generated code, explain in your own words the underlying concept — not just what the code says.

---
## A Note on Variance

RL training has two independent sources of randomness:

**1. Training randomness** — random weight initialization, action sampling, and minibatch ordering. Two runs with different seeds can converge to meaningfully different policies.

**2. Evaluation randomness** — each episode starts from a different random initial state. A good policy on average may still fail on specific seeds.

This is why we evaluate over multiple episodes and report mean ± std. If you see high variance in your eval curve, that is expected — it is the nature of RL, not a bug.

---
## Bonus — Fix the Reward 💻☁️

You diagnosed the reward hacking problem in Step 10. Now fix it.

### The task

Write a **reward wrapper** — a class that wraps the environment and modifies the reward signal at each step. The training loop doesn't need to change at all; you just swap in your wrapped environment.

```python
class FixedRewardEnv:
    def __init__(self, env):
        self.env = env
        self.observation_space = env.observation_space
        self.action_space      = env.action_space

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # TODO: modify reward here
        return obs, reward, terminated, truncated, info

    def close(self):
        return self.env.close()
```

### Requirements

1. **Implement at least one reward modification** from your proposals in Step 10. Use the observation to compute your additional term — check the [Gymnasium docs](https://gymnasium.farama.org/environments/mujoco/half_cheetah/) for which obs indices correspond to torso height and angle.

2. **Justify your choice** in the markdown cell below: what does it penalize or reward, why do you expect it to fix the backflop, and what unintended effects might it introduce?

3. **Retrain from scratch** using your wrapped environment. Did the cheetah learn to run upright? Did eval return go up or down vs the original?

### Hints

- `obs[0]` is the z-coordinate (height) of the torso. Upright ≈ 0.0; flipped is lower.
- `obs[1]` is the torso angle. Upright ≈ 0; flipped ≈ ±π.
- To wrap the `VecEnv`, pass `FixedRewardEnv(gym.make(ENV_NAME))` instead of `gym.make(ENV_NAME)` when building the env list.
- Start with a small penalty coefficient (e.g. 0.5) — a very large penalty can destabilize training.

### My reward modification

*Explain your choice here: what does it penalize/reward, why should it fix the backflop, and what new problems might it cause?*

In [ ]:
class FixedRewardEnv:
    """
    TODO: implement your reward wrapper here.
    Wrap the environment so that each step returns a modified reward
    that discourages the backflop behavior.
    """
    def __init__(self, env):
        self.env = env
        self.observation_space = env.observation_space
        self.action_space      = env.action_space

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # TODO: compute your shaping term and add it to reward
        # torso_height = obs[0]
        # torso_angle  = obs[1]
        # shaping = ...
        # reward = reward + shaping
        return obs, reward, terminated, truncated, info

    def close(self):
        return self.env.close()


class FixedVecEnv(VecEnv):
    """VecEnv that wraps each environment with FixedRewardEnv."""
    def __init__(self, env_name, num_envs, seed=0):
        # TODO: build self.envs as a list of FixedRewardEnv(gym.make(env_name))
        # and seed each one as in the original VecEnv
        self.envs     = None  # replace
        self.num_envs = num_envs


# ── Retrain with the fixed reward ────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

fixed_policy    = ActorCritic(OBS_DIM, ACT_DIM, HIDDEN).to(DEVICE)
fixed_optimizer = optim.Adam(fixed_policy.parameters(), lr=LR, eps=1e-5)
fixed_buffer    = RolloutBuffer(ROLLOUT_STEPS, NUM_ENVS, OBS_DIM, ACT_DIM, DEVICE)
fixed_vec_env   = FixedVecEnv(ENV_NAME, NUM_ENVS, seed=42)

obs  = torch.tensor(fixed_vec_env.reset(), dtype=torch.float32, device=DEVICE)
done = torch.zeros(NUM_ENVS, device=DEVICE)

fixed_episode_returns = []
fixed_eval_steps      = []
fixed_eval_means      = []
fixed_eval_stds       = []
steps_done            = 0
current_ep_ret        = torch.zeros(NUM_ENVS, device=DEVICE)

pbar = tqdm(range(total_rollouts), desc='PPO (fixed reward)')
for rollout_idx in pbar:
    if ANNEAL_LR:
        frac = 1.0 - rollout_idx / total_rollouts
        for g in fixed_optimizer.param_groups:
            g['lr'] = LR * frac

    fixed_buffer.reset()
    fixed_policy.eval()
    rollout_ep_returns = []

    with torch.no_grad():
        for step in range(ROLLOUT_STEPS):
            action, log_prob, value = fixed_policy.act(obs)
            next_obs_np, reward_np, done_np = fixed_vec_env.step(action.cpu().numpy())
            reward    = torch.tensor(reward_np,   dtype=torch.float32, device=DEVICE)
            next_obs  = torch.tensor(next_obs_np, dtype=torch.float32, device=DEVICE)
            next_done = torch.tensor(done_np,     dtype=torch.float32, device=DEVICE)
            fixed_buffer.store(obs, action, reward, done, log_prob, value)
            current_ep_ret += reward
            for i, d in enumerate(done_np):
                if d:
                    rollout_ep_returns.append(current_ep_ret[i].item())
                    current_ep_ret[i] = 0.0
            obs  = next_obs
            done = next_done
        _, _, next_value = fixed_policy.act(obs)

    advantages, returns = fixed_buffer.compute_gae(next_value, done)
    steps_done += ROLLOUT_STEPS * NUM_ENVS
    fixed_policy.train()
    pl, vl, ent = ppo_update(fixed_policy, fixed_optimizer, fixed_buffer, advantages, returns)
    if rollout_ep_returns:
        fixed_episode_returns.append(np.mean(rollout_ep_returns))

    if (rollout_idx + 1) % LOG_INTERVAL == 0:
        avg = np.mean(fixed_episode_returns[-20:]) if fixed_episode_returns else float('nan')
        pbar.set_postfix({'steps': f'{steps_done/1e6:.2f}M', 'ret': f'{avg:.0f}'})

    if (rollout_idx + 1) % EVAL_INTERVAL == 0:
        m, s = evaluate(fixed_policy, ENV_NAME, num_episodes=EVAL_EPISODES)
        fixed_eval_means.append(m); fixed_eval_stds.append(s); fixed_eval_steps.append(steps_done)
        print(f'\n[{steps_done/1e6:.2f}M steps] fixed eval: {m:.1f} ± {s:.1f}')

fixed_vec_env.close()
torch.save(fixed_policy.state_dict(), 'ppo_policy_fixed.pt')
print('Saved ppo_policy_fixed.pt')

In [ ]:
# ── Compare original vs fixed eval return ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].errorbar(eval_steps,       eval_means,       yerr=eval_stds,
                 fmt='o-', capsize=4, label='original reward')
axes[0].errorbar(fixed_eval_steps, fixed_eval_means, yerr=fixed_eval_stds,
                 fmt='s--', capsize=4, label='fixed reward')
axes[0].set_xlabel('Environment steps'); axes[0].set_ylabel('Eval return')
axes[0].set_title('Eval Return: Original vs Fixed Reward')
axes[0].legend()

axes[1].plot(fixed_episode_returns, alpha=0.3, label='episode return (fixed)')
W = 20
if len(fixed_episode_returns) >= W:
    smooth = np.convolve(fixed_episode_returns, np.ones(W)/W, mode='valid')
    axes[1].plot(range(W-1, len(fixed_episode_returns)), smooth, label=f'{W}-ep avg')
axes[1].set_xlabel('Completed episodes'); axes[1].set_ylabel('Return')
axes[1].set_title('Training Return (fixed reward)'); axes[1].legend()

plt.tight_layout()
plt.savefig('ppo_comparison.png', dpi=120)
plt.show()

print(f'Original final eval:  {eval_means[-1]:.1f} ± {eval_stds[-1]:.1f}')
print(f'Fixed    final eval:  {fixed_eval_means[-1]:.1f} ± {fixed_eval_stds[-1]:.1f}')

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
fixed_policy.eval()

def fixed_ppo_step(obs):
    with torch.no_grad():
        obs_t   = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        mean, _ = fixed_policy(obs_t)
        return mean.squeeze(0).cpu().numpy()

print('--- Original (backflop) policy ---')
record_policy(ppo_step, ENV_NAME, 'ppo_policy.mp4')
display(Video('ppo_policy.mp4', embed=True, width=600))

print('\n--- Fixed reward policy ---')
record_policy(fixed_ppo_step, ENV_NAME, 'ppo_policy_fixed.mp4')
display(Video('ppo_policy_fixed.mp4', embed=True, width=600))

In [ ]:
# ── Bonus 2 sandbox ──────────────────────────────────────────────────────────
# Modify whatever you like below. Keep the original policy above intact.
# Define your improved policy with a new name, e.g. bonus_policy.

# TODO: implement your improved recipe here

# torch.save(bonus_policy.state_dict(), 'ppo_policy_bonus.pt')
# print('Bonus policy saved.')